# ಪಾಠ 10 - ಉತ್ಪಾದನಿಯಲ್ಲಿ AI ಏಜೆಂಟ್‌ಗಳು

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework (Python) ಬಳಸಿ AI ಏಜೆಂಟ್‌ಗಳ **ಉತ್ಪಾದನಾ ಮಾದರಿಗಳನ್ನು** ಕಲಿಯೋಿರಿ. ನಾವು ಒಳಗೊಂಡಿದ್ದೇವೆ:

- **ನಿರೀಕ್ಷಣೀಯತೆ** — ಏಜೆಂಟ್ ಸಂವಹನಗಳಿಗೆ ಸಮಯ ಗಡಿವಳಿಕೆ ಮತ್ತು ಲಾಗಿಂಗ್ ಸೇರಿಸುವುದು  
- **ಮೌಲ್ಯಮಾಪನ** — ಪ್ರತಿಕ್ರಿಯೆಯ ಗುಣಮಟ್ಟವನ್ನು ಅಂಕಗಳಿಸಲು ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ಬಳಸುವುದು  
- **ಕೋಸ್ಟ್ ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಅತಿವೃದ್ದಿ ಮತ್ತು ಮಾದರಿ ಆಯ್ಕೆಗಾಗಿ ತಂತ್ರಗಳು  

ದೃಶ್ಯವು **ಪ್ರಯಾಣ ಏಜೆಂಟ್** ಆಗಿದ್ದು, ಇದು ಬಳಕೆದಾರರಿಗೆ ಪ್ರವಾಸ ಯೋಜಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತದೆ, ಮೇಲ್ಭಾಗದಲ್ಲಿ ಹತ್ತಿರ ನಿಗಾ ಮತ್ತು ಮೌಲ್ಯಮಾಪನ ವ್ಯವಸ್ಥೆಯನ್ನು ಹೊಂದಿದೆ.


## ಸೆಟ್‌ಅಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಉತ್ಪಾದನಾ ಪರಿಗಣನೆಗಳು

ಕೃತಕ ಬುದ್ಧಿಮತ್ತೆ ಏಜೆಂಟ್‌ಗಳನ್ನು ಪ್ರೋಟೋಟೈಪ್‌ಗಳಿಂದ ಉತ್ಪಾದನೆಗೆ ಸ್ಥಳಾಂತರಿಸುವುದು ಮೂರು ಪ್ರಮುಖ ಅಂಶಗಳಿಗೆ ಶ್ರದ್ಧೆ ನೀಡಬೇಕಾಗುತ್ತದೆ:

1. **ನಿರೀಕ್ಷಣೀಯತೆ** — ಏಜೆಂಟ್ ಏನು ಮಾಡುತ್ತಿದೆ, ಅದರ ಸಮಯ ಎಷ್ಟು ತೆಗೆದುಕೊಳ್ಳುತ್ತದೆ ಮತ್ತು ಯಾವ ಯಂತ್ರಗಳನ್ನು ಕರೆ ಮಾಡುತ್ತದೆ ಎಂದು ನೀವು ಗೋಚರಿಸಬೇಕಾಗುತ್ತದೆ. ಅನುಕ್ರಮಣ ಮತ್ತು ಲಾಗಿಂಗ್ ಇರುವದು ಇಲ್ಲದೆ, ಉತ್ಪಾದನಾ ಸಮಸ್ಯೆಗಳನ್ನು ನಿದಾನಿಸಲು ಸಾಧ್ಯವಿಲ್ಲ.

2. **ಮೌಲ್ಯಮಾಪನ** — ಸ್ವಯಂಚಾಲಿತ ಗುಣಮಟ್ಟ ಪರಿಶೀಲನೆಗಳು ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಗಳು ನಿಖರ, ಸಂಪೂರ್ಣ ಮತ್ತು ಸಹಾಯಕವಾಗಿರುವುದನ್ನು ಕಾಲಕಾಲಕ್ಕೆ ಖಚಿತಪಡಿಸುತ್ತವೆ. ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್ ನಿರ್ದಿಷ್ಟ ನಿಯಮಾವಳಿಗಳ ವಿರುದ್ಧ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಅಂಕಿಅಂಶ ನೀಡುತ್ತದೆ.

3. **ವೆಚ್ಚ ನಿರ್ವಹಣೆ** — ಟೋಕನ್ ಉಪಯೋಗವು ನೇರವಾಗಿ ವೆಚ್ಚವನ್ನು ಪ್ರಭಾವಿಸುತ್ತದೆ. ಪ್ರಾಂಪ್ಟ್ ಚೇತನಗೊಳಿಸುವಿಕೆ, ಮಾದರಿ ಆಯ್ಕೆ ಮತ್ತು ಕ್ಯಾಶಿಂಗ್ ಹಂಗಾಮೆಗಳು ಗುಣಮಟ್ಟಕ್ಕೆ ತಾಕೀತು ಕೊಡುವದೆ ಇಲ್ಲದೆ ವೆಚ್ಚಗಳನ್ನು ನಿಯಂತ್ರಣದಲ್ಲಿಡಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.


## ಒಂದು ಗಮನಾರ್ಹ ಏಜೆಂಟ್ ಅನ್ನು ನಿರ್ಮಿಸುವುದು

ನಾವು ಪ್ರಯಾಣ ಸಾಧನಗಳನ್ನುนิರೋಧಿಸುತ್ತೇವೆ ಮತ್ತು ಏಜೆಂಟ್ ಕರೆ ಅನ್ನು ಸಮಯ ಮಟ್ಟದಲ್ಲಿ ಇಡುತ್ತೇವೆ ώστε ನಾವು ವಿಳಂಬವನ್ನು ನೋಡಬಹುದು. ಉತ್ಪಾದನೆಯಲ್ಲಿ ನೀವು OpenTelemetry ಅಥವಾ ಹಾಗೆಯೇ ಟ್ರೇಸಿಂಗ್ ಬ್ಯಾಕೆಂಡ್‌ ಆಗಿ ಸಂಯೋಜಿಸಲು ಸಾಧ್ಯ.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ಮೌಲ್ಯಮಾಪನ ಮಾದರಿಗಳು

ಸಾಮಾನ್ಯ ಉತ್ಪಾದನಾ ಮಾದರಿ ಎಂದರೆ ಎರಡನೇ ಏಜೆಂಟ್ ಅವರನ್ನು **ಮೌಲ್ಯಮಾಪಕ**ನಾಗಿ ಬಳಸುವುದು. ಮೌಲ್ಯಮಾಪಕವು ಮೊದಲನೇ ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆಯಂತಹ ಪೂರ್ವನಿರ್ಧಾರಿತ ಮಾನದಂಡಗಳ ವಿರುದ್ಧ ಅಂಕಂಗೊಳಿಸುತ್ತದೆ.

ಇದರ ಮೂಲಕ ಸಾಧ್ಯವಾಗುವುದು:
- ಬಳಕೆದಾರರಿಗೆ ಪ್ರತಿಕ್ರಿಯೆಗಳು ತಲುಪುವ ಮೊದಲು ಸ್ವಯಂಚಾಲಿತ ಗುಣಮಟ್ಟದ ಗೇಟುಗಳು
- ಪ್ರಾಂಪ್ಟ್‌ಗಳು ಅಥವಾ ಮಾದರಿಗಳು ಬದಲಾಗುವಾಗ ಹಿಂತಲಗುವಿಕೆ ಪತ್ತೆಮಾಡುವುದು
- ಏಜೆಂಟ್ ಕಾರ್ಯಕ್ಷಮತೆಯ ನಿರಂತರ ಗಮನವಿಡುವಿಕೆ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ವೆಚ್ಚ ನಿರ್ವಹಣಾ ತಂತ್ರಗಳು

ಉತ್ಪಾದನಾ AI ಏಜೆಂಟ್‌ಗಳಿಗಾಗಿ ವೆಚ್ಚಗಳನ್ನು ನಿಯಂತ್ರಿಸುವುದು ಅತ್ಯಂತ ಮುಖ್ಯವಾಗಿದೆ. ಇಲ್ಲಿ ಪ್ರಮುಖ ತಂತ್ರಗಳು ಇವೆ:

| ತಂತ್ರ | ವಿವರಣೆ |
|---|---|
| **ಪ್ರಾಂಪ್ಟ್ ಆಪ್ಟಿಮೈಸೇಶನ್** | ವ್ಯವಸ್ಥೆಯ ಸೂಚನೆಗಳನ್ನು ಸಂಕ್ಷೇಪವಾಗಿ ಇಡಿ. ಇನ್ಪುಟ್ ಟೋಕನ್ಗಳನ್ನು ಕಡಿಮೆಗೆ ತರುವುದಕ್ಕಾಗಿ ಅನಾವಶ್ಯಕ ಸಂದರ್ಭಗಳನ್ನು ತೆಗೆದುಹಾಕಿ. |
| **ಮодель ಆಯ್ಕೆ** | ಸರಳ ಕೆಲಸಗಳಿಗೆ ತರಗತಿ ಅಥವಾ ಹೊರತೆಗೆಯುವಿಕೆಗೆ ಸಣ್ಣ, saste ಮಾದರಿಗಳನ್ನು (ಉದಾ. GPT-4o-mini) ಬಳಸಿ, ಪ್ರಮುಖ ನಿರ್ದಿಷ್ಟ ನಿರ್ಣಯಕ್ಕಾಗಿ ದೊಡ್ಡ ಮಾದರಿಗಳನ್ನು ಮೀಸಲಿಡಿ. |
| **ಕ್ಯಾಚಿಂಗ್** | ಸಾಧನ ಫಲಿತಾಂಶ ಮತ್ತು ಸಾಂದರ್ಭಿಕ ಪ್ರಶ್ನೆಗಳನ್ನು ಕ್ಯಾಚ್ ಮಾಡಿ ಮರುಪಾವತಿಯಾಗುವ API ಕರೆಗಳನ್ನು ತಪ್ಪಿಸಿ. |
| **ಟೋಕನ್ ಬಜೆಟ್‌ಗಳು** | ಅನಿರೀಕ್ಷಿತವಾಗಿ ಉದ್ದವಾದ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ತಡೆಯಲು `max_tokens` ಮಿತಿಯನ್ನು ನಿಗದಿಪಡಿಸಿ. |
| **ಬ್ಯಾಚಿಂಗ್** | ಸಾಧ್ಯವಿದ್ದಲ್ಲಿ ಒಂದೇ API ಕರೆಗಾಗಿ ಬಹು ಬಳಕೆದಾರರು ಪ್ರಶ್ನೆಗಳನ್ನು ಗುಂಪುಮಾಡಿ. |

ಅಭ್ಯಾಸದಲ್ಲಿ, ಹಂತಬಂಧಿತ ವಿಧಾನ ಚೆನ್ನಾಗಿದೆ: ಸರಳ ವಿನಂತಿಗಳನ್ನು ವೇಗವಾದ, ಕಡಿಮೆ ದರದ ಮಾದರಿಗೆ ಮಾರ್ಗದರ್ಶನ ಮಾಡಿ, ಕೇವಲ ಸಂಕೀರ್ಣ ಪ್ರಶ್ನೆಗಳನ್ನು ಹೆಚ್ಚು ಸಾಮರ್ಥ್ಯವುಳ್ಳ ಮಾದರಿಗೆ ಏರಿಸಿ.


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಈ ಕೆಳಗಿನವುಗಳನ್ನು ಕಲಿತಿರಿ:

1. **ಓಬ್ಸರ್‌ವೆಬಿಲಿಟಿ ಸೇರಿಸುವುದು** ಆಜೆಂಟ್ ಸಂವಹನಗಳಿಗೆ ಸಮಯನಿರ್ದೇಶನ ಮತ್ತು ಲಾಗ್ ಮಾಡಿದ ಮೂಲಕ, ಟ್ರೇಸಿಂಗ್ ಮತ್ತು ಮಾನಿಟರಿಂಗ್ ಮೊದಲನೇ ಹಂತವನ್ನು ರೂಪಿಸುವುದು.
2. **ಆಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಮೌಲ್ಯಮಾಪನ ಮಾಡುವುದು** ಸ್ವಯಂಚಾಲಿತವಾಗಿ, ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆ ಅಂಶಗಳನ್ನು ಅಂಕದ ಮೂಲಕ ಅಳೆಯುವ ಮೌಲ್ಯಮಾಪಕ ಆಜೆಂಟ್ ಬಳಸಿ.
3. **ಖರ್ಚು ನಿರ್ವಹಣೆ** ಪ್ರಾಂಪ್ಟ್ ಅೋಪ್ಟಿಮೈಜೇಶನ್, ಮಾದರಿ ಆಯ್ಕೆ, ಕ್ಯಾಶಿಂಗ್ ಮತ್ತು ಟೊಕೆನ್ ಬಜೆಟ್‌ಗಳ ಮೂಲಕ.

ಈ ಉತ್ಪಾದನಾ ಮಾದರಿಗಳು ನಿಮ್ಮ AI ಆಜೆಂಟ್‌ಗಳ ವಿಶ್ವಾಸಾರ್ಹತೆ, ಅಳೆಯಬಹುದಾದತೆ ಮತ್ತು ವೆಚ್ಚದ ಪರಿಣಾಮಕಾರಿತ್ವವನ್ನು ಖಚಿತಪಡಿಸುತ್ತವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
